# Silver — CRM Product Info
Product master data from the CRM.

`bronze.crm_prd_info` → `silver.crm_products`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_prd_info")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Parse the product key
`prd_key` looks like `CO-RF-FR-R92B-58`. The first 5 chars (`CO-RF` → `CO_RF`) are the category id used by the ERP; the rest is the real product number.

In [ ]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

### Default missing cost to 0

In [ ]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

### Normalize product line

In [ ]:
df = df.withColumn(
    "prd_line",
    F.when(F.upper(col("prd_line")) == "M", "Mountain")
     .when(F.upper(col("prd_line")) == "R", "Road")
     .when(F.upper(col("prd_line")) == "S", "Other Sales")
     .when(F.upper(col("prd_line")) == "T", "Touring")
     .otherwise("n/a")
)

### Cast dates

In [ ]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.crm_products")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10;